# SWE015 · Real-Time Chat Moderation: training on Google Colab

This notebook runs the whole pipeline: **data → 4 models → quantization → evaluation → app**.

1. **Runtime → Change runtime type → T4 GPU** (BERT fine-tuning needs the GPU; everything else runs on the CPU).
2. Run the cells from top to bottom. The whole notebook takes about 20-30 minutes.
3. **Every number for your slides comes from the outputs of this notebook.** Copy them from here or from `results/`.

## 1 · Get the code
Option A: clone your GitHub repository (after you push it). Option B: upload the project zip.

In [ ]:
# Option A - clone from GitHub (change the URL)
REPO_URL = "https://github.com/ghaithpph1/chat-moderation.git"
import os
if not os.path.exists("/content/chat-moderation") and "<your-username>" not in REPO_URL:
    !git clone {REPO_URL} /content/chat-moderation

In [ ]:
# Option B - upload chat-moderation.zip instead (skip this cell if you used option A)
import os
if not os.path.exists("/content/chat-moderation"):
    from google.colab import files
    uploaded = files.upload()                      # choose chat-moderation.zip
    !unzip -q -o chat-moderation.zip -d /content/

In [ ]:
%cd /content/chat-moderation
!pip install -q -r requirements-train.txt
!nvidia-smi -L
!python -c "import torch, transformers, onnxruntime, gensim, sklearn; print('torch', torch.__version__, '| transformers', transformers.__version__, '| onnxruntime', onnxruntime.__version__, '| gensim', gensim.__version__, '| sklearn', sklearn.__version__, '| GPU:', torch.cuda.is_available())"

## 2 · Data
OLID / OffensEval-2019 English tweets (via TweetEval). Official train / validation / test splits. The **test split is the single held-out test set for every model**. Note the sizes and the share of offensive messages for the *Data* slide.

In [ ]:
!python -m src.data

In [ ]:
# Normalization examples for the Data slide (before -> after)
!python -m src.normalize

## 3 · Model 1: lexicon + minimum edit distance (rule baseline, area B)

In [ ]:
!python -m src.lexicon

## 4 · Model 2: TF-IDF + logistic regression (classical baseline, area B)
C is chosen on the validation set.

In [ ]:
!python -m src.tfidf_model

## 5 · Model 3: our own word2vec + feed-forward network trained with SGD (areas B + A)

In [ ]:
!python -m src.w2v_ffn

## 6 · Model 4: fine-tune BERT (area C)
About 5-10 minutes on a T4. The best epoch (validation macro-F1) is saved to `models/bert_fp32`.

In [ ]:
!python -m src.bert_model

## 7 · Model 5: quantize BERT to INT8 (area D)
Export to ONNX, then ONNX Runtime dynamic quantization (plus PyTorch dynamic INT8 as an extra row).

In [ ]:
!python -m src.quantize

## 8 · Evaluate every model on the same test set (R3 + R4)
Latency is measured on the **CPU**, one message at a time (10 warm-up runs + 200 timed runs). The hardware line is printed under the table. Keep this machine's numbers together; don't mix them with numbers from another machine.

In [ ]:
!python -m src.evaluate

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/results.md", encoding="utf-8").read()))

### Failure examples for the *Limitations* slide
The most confident mistakes of the app's model come first.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 160)
pd.read_csv("results/errors_bert_int8_onnx.csv").head(10)

In [ ]:
# Where does BERT INT8 get it right but TF-IDF wrong (and the other way around)?
p = pd.read_csv("results/predictions_test.csv")
truth = p["label"].map({0: "clean", 1: "offensive"})
bert_only = p[(p["bert_int8_onnx_pred"] == truth) & (p["tfidf_pred"] != truth)]
tfidf_only = p[(p["tfidf_pred"] == truth) & (p["bert_int8_onnx_pred"] != truth)]
print(len(bert_only), "test messages only BERT INT8 gets right;", len(tfidf_only), "only TF-IDF gets right")
bert_only[["text", "label", "tfidf_pred", "bert_int8_onnx_pred"]].head(5)

In [ ]:
# Training curves figure for the 'how each model was trained' slide
!python scripts/plot_training_curves.py
from IPython.display import Image
Image("results/training_curves.png")

## 9 · Try the app here
This prints a public `gradio.live` link. Type new messages, switch models, and watch the latency log. **Stop the cell** (■) to continue with the notebook.

For the video, run the app on your own laptop (`python app.py`) after step 10. That shows it running on a normal CPU.

In [ ]:
!python app.py --share

## 10 · Save the weights (they must NOT go into Git)
**Hugging Face Hub (recommended)**: create a free account and a token with *write* access at https://huggingface.co/settings/tokens. The app downloads the weights from here automatically.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_REPO = "ghaithpph/chat.moderation.models"   # change this
!python scripts/upload_models.py --repo {HF_REPO}

In [ ]:
# Backup copy in your Google Drive (optional)
from google.colab import drive
drive.mount("/content/drive")
!zip -q -r /content/drive/MyDrive/chat_moderation_models.zip models -x "models/bert_onnx_fp32/*"

## 11 · Download `results/` and put it in your repository
Then set `MODEL_REPO` in `config.py` to your Hugging Face repo, paste `results/results.md` into the README, and push to GitHub.

In [ ]:
!zip -q -r results.zip results
from google.colab import files
files.download("results.zip")